# Inspect saved data and models
Run `01_run.ipynb` (or its Python CLI equivalent) first. This notebook does not train. Use `restore` to inspect a GPU-trained checkpoint on CPU as well. Move the entire output directory together: artifact references are relative.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "settings.py").is_file() and (p / "nnpd").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from settings import make_config
from nnpd import execute, plan, restore, load_experiment
from nnpd.results import runs

In [ ]:
config = make_config("smoke")
experiment = load_experiment(config["application"])
records = runs(config["output"])
assert records, "Run 01_run.ipynb or python run.py run --profile smoke first."
context = restore(records[0]["path"], experiment, device="cpu")
context.model

In [ ]:
observations = context.require("observations")
inference = context.require("inference")
{"truth_shape": observations.array("truth").shape,
 "observation_shape": observations.array("observations").shape,
 "mode_shape": inference.array("ratio_mode").shape,
 "metadata": inference.metadata}

## Call a metric or plot directly
Every hook is an ordinary Python function. Arrays are numeric `.npy` files, normally opened as read-only memory maps. Full candidate scores are retained only when requested in settings; the default retains diagnostic cases and all per-case summaries.

In [ ]:
hook = experiment.metrics()["coverage"]
hook.compute(context, context.dependencies(hook.needs))

In [ ]:
from IPython.display import display
from matplotlib import pyplot as plt
hook = experiment.plots()["inference"]
figures = hook.draw(context, context.dependencies(hook.needs))
display(next(iter(figures.values())))
for figure in figures.values():
    plt.close(figure)

In [ ]:
from nnpd.results import export_csv
export_csv(config["output"], Path(config["output"]) / "comparison.csv",
           {"prior": "member.prior", "dimension": "config.problem.dimension",
            "auc": "metrics.classification.auc", "mean_absolute_bias": "metrics.bias.mean_absolute"})